In [3]:
import os
import numpy as np
from urllib.request import urlretrieve
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain



In [4]:
import sys
print(sys.executable)

/Users/egegulunay/PROJECTS/pdf_reader/.venv/bin/python


In [5]:
"""os.makedirs("us_census",exist_ok=True)

files = [
    "https://www.census.gov/content/dam/Census/library/publications/2022/demo/p70-178.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-017.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-016.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-015.pdf",

]

for url in files:
    file_path = os.path.join("us_census",url.rpartition("/")[2])
    urlretrieve(url, file_path)
    
    
HTTPError: HTTP Error 403: Forbidden
"""

'os.makedirs("us_census",exist_ok=True)\n\nfiles = [\n    "https://www.census.gov/content/dam/Census/library/publications/2022/demo/p70-178.pdf",\n    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-017.pdf",\n    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-016.pdf",\n    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-015.pdf",\n\n]\n\nfor url in files:\n    file_path = os.path.join("us_census",url.rpartition("/")[2])\n    urlretrieve(url, file_path)\n\n\nHTTPError: HTTP Error 403: Forbidden\n'

In [6]:
import os
import requests

os.makedirs("us_census", exist_ok=True)

files = [
    "https://www.census.gov/content/dam/Census/library/publications/2022/demo/p70-178.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-017.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-016.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-015.pdf",
]

headers = {
    "User-Agent": "Mozilla/5.0"
}

for url in files:
    filename = url.rpartition("/")[2]
    file_path = os.path.join("us_census", filename)

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {file_path}")

Downloaded: us_census/p70-178.pdf
Downloaded: us_census/acsbr-017.pdf
Downloaded: us_census/acsbr-016.pdf
Downloaded: us_census/acsbr-015.pdf


In [7]:
loader = PyPDFDirectoryLoader("us_census")
documents = loader.load()

print(f"Total pages: {len(documents)}")
print(documents[0].metadata)
print(documents[0].page_content[:500])

Total pages: 63
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census/acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}
Health Insurance Coverage Status and Type 
by Geography: 2021 and 2022
American Community Survey Briefs
ACSBR-015
Issued September 2023
Douglas Conway and Breauna Branch
INTRODUCTION
Demographic shifts as well as economic and govern-
ment policy changes can affect people’s access to 
health coverage. For example, between 2021 and 2022, 
the labor market continued to improve, which may 
have affected private coverage in the United States 
during that time.
1 Public policy changes included 
the re


In [8]:
print(type(documents))
print(type(documents[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

In [10]:
print(len(documents))  # sayfa sayısı
print(len(chunks))     # chunk sayısı
print(chunks[0].page_content)

63
398
Health Insurance Coverage Status and Type 
by Geography: 2021 and 2022
American Community Survey Briefs
ACSBR-015
Issued September 2023
Douglas Conway and Breauna Branch
INTRODUCTION
Demographic shifts as well as economic and govern-
ment policy changes can affect people’s access to 
health coverage. For example, between 2021 and 2022, 
the labor market continued to improve, which may 
have affected private coverage in the United States 
during that time.
1 Public policy changes included 
the renewal of the Public Health Emergency, which 
allowed Medicaid enrollees to remain covered under 
the Continuous Enrollment Provision.
2 The American 
Rescue Plan (ARP) enhanced Marketplace premium


In [11]:
def average_length(docs):
    total_characters = sum(len(doc.page_content) for doc in docs)
    return total_characters // len(docs)

avg_page_length = average_length(documents)
avg_chunk_length = average_length(chunks)

print(f"Pages        : {len(documents)}")
print(f"Chunks       : {len(chunks)}")
print(f"Avg page len : {avg_page_length}")
print(f"Avg chunk len: {avg_chunk_length}")

Pages        : 63
Chunks       : 398
Avg page len : 3840
Avg chunk len: 624


In [12]:
embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5", 
    model_kwargs={"device": "cpu"},        
    encode_kwargs={"normalize_embeddings": True}  
)

/var/folders/wg/_5btzj_n7hb27_yh7r39tlm40000gn/T/ipykernel_3526/727412182.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9574.72it/s]


In [13]:
sample_embedding = np.array(embeddings.embed_query(chunks[0].page_content))

print("Embedding dimension:", len(sample_embedding))
print(sample_embedding[:10])

Embedding dimension: 384
[-0.07508394 -0.01188476 -0.0314888   0.02940387  0.05034871  0.05624264
 -0.01690789  0.03468882 -0.09790616 -0.02528042]


In [14]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [15]:
# Test Retrieval
query = "What is health insurance coverage?"

relevant_documents = vectorstore.similarity_search(query)

print(relevant_documents[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC).


In [16]:
# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

In [17]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY bulunamadı!")

In [18]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=api_key
)

In [19]:
response  = llm.invoke(query)

print(response.content)

Health insurance coverage refers to a type of insurance that pays for medical and surgical expenses incurred by the insured. It can also provide coverage for other types of health-related costs, such as preventive care, mental health services, and prescription medications. Here are some key components of health insurance coverage:

1. **Types of Plans**: Health insurance can come in various forms, including employer-sponsored plans, government programs (like Medicare and Medicaid), and individual plans purchased through insurance marketplaces.

2. **Premiums**: This is the amount you pay, usually monthly, to maintain your health insurance coverage.

3. **Deductibles**: This is the amount you must pay out-of-pocket for healthcare services before your insurance begins to pay.

4. **Copayments and Coinsurance**: These are the costs you share with your insurance after meeting your deductible. A copayment is a fixed amount you pay for a specific service, while coinsurance is a percentage of

In [20]:
prompt = ChatPromptTemplate.from_template("""
use the following context to answer the user's question.

If the answer is not in the context, say that you don't know 

Context:
{context}

Question:
{input}                                                                                                                     
 """)

In [ ]:
document_chain = create_stuff_documents_chain(
    llm, 
    prompt)

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [23]:
retrieval_chain.invoke(
    {"input":query}
)

{'input': 'What is health insurance coverage?',
 'context': [Document(id='667bf4c3-e5e1-40bf-b98c-b6dcee50d227', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census/acsbr-015.pdf', 'total_pages': 18, 'page': 1, 'page_label': '2'}, page_content='2 U.S. Census Bureau\nWHAT IS HEALTH INSURANCE COVERAGE?\nThis brief presents state-level estimates of health insurance coverage \nusing data from the American Community Survey (ACS). The  \nU.S. Census Bureau conducts the ACS throughout the year; the \nsurvey asks respondents to report their coverage at the time of \ninterview. The resulting measure of health insurance coverage, \ntherefore, reflects an annual average of current comprehensive \

In [24]:
import gradio as gr

In [28]:
def ask_pdf(question):
    response = retrieval_chain.invoke({"input":question})
    answer = response["answer"]

    sources = ""
    for i, doc in enumerate(response["context"], start=1):
        sources += f"\nSource {i}: {doc.metadata['source']} - Page {doc.metadata.get('page_label', doc.metadata.get('page'))}\n"

    return answer,sources

demo = gr.Interface(
    fn = ask_pdf,
    inputs= gr.Textbox(label="Ask a question"),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Sources")
    ],
    title="RAG PDF Chatbot",
    description="Ask questions about the loaded PDF documents."
)

demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


/Users/egegulunay/PROJECTS/pdf_reader/.venv/lib/python3.11/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
